# BTXRD WSSS — Final Thesis Run (WSSS pipeline + fully-supervised oracle comparison)

This notebook is intentionally split into small, inspectable stages. It calls the current repository scripts rather
than reimplementing the production pipeline in hidden notebook code.

```text
tumor_type (10 image-level classes)
 -> DenseNet121 CE, 320 px
 -> LayerCAM denseblock2/3/4, weights 0.2/0.3/0.5
 -> class-vs-normal contrast
 -> CAM percentiles 85/90/95
 -> up to 3 CAM components
 -> box + positive/negative points
 -> SAM ViT-B, 512 px
 -> box_point + point + box prompt ensemble
 -> coverage_mass_sam, component_topk=3
 -> tumor morphology and pseudo-mask
```

`predicted` and `ground_truth` CAM protocols are always written to separate directories. Polygon GT is loaded only in
explicit diagnostic/evaluation cells and is never passed to classifier, CAM, prompts, SAM, candidate selection, or
post-processing.


This is the final run notebook for the thesis: end-to-end WSSS pipeline (classifier -> LayerCAM -> SAM -> pseudo-mask -> U-Net) plus the fully-supervised oracle baseline (Image -> GT -> U-Net) trained and evaluated the same way, with a final side-by-side comparison (section 10b). The canonical locked recipe is `btxrd_best`; any alternative is an explicitly separate experiment.

## Execution order and switches

1. Runtime/repository/dataset audit
2. Split and leakage checks
3. Ground-truth visualization (diagnostic only)
4. Classifier training and CAM snapshots
5. Single-image CAM/morphology/prompt/SAM debug trace
6. Separate full validation runs for `predicted` and `ground_truth`
7. Metrics, oracle decomposition, qualitative panels, and artifact manifest

Expensive stages are controlled in Cell 1. No test-set tuning is performed.


In [ ]:
# Cell 0 - Kaggle bootstrap: clone source, install dependencies, download SAM, and prepare split manifest
from pathlib import Path
import os
import shutil
import subprocess
import sys
import urllib.request

KAGGLE_WORKING_BOOTSTRAP = Path("/kaggle/working")
KAGGLE_INPUT_BOOTSTRAP = Path("/kaggle/input")
REPOSITORY_URL = "https://github.com/itsthang333/Thesis.git"
REPOSITORY_BRANCH = "pipeline"
DEFAULT_DATASET_ROOT = Path("/kaggle/input/datasets/wanwin/data-btxrd/BTXRD")
BOOTSTRAP_REPO_ROOT = KAGGLE_WORKING_BOOTSTRAP / "Thesis"

# A locally opened repository is useful for development; a Kaggle notebook uploaded
# by itself has no project/ directory and therefore clones the public pipeline branch.
if (Path.cwd() / "project").is_dir() and (Path.cwd() / ".git").is_dir():
    BOOTSTRAP_REPO_ROOT = Path.cwd().resolve()
elif not (BOOTSTRAP_REPO_ROOT / "project").is_dir():
    if BOOTSTRAP_REPO_ROOT.exists():
        raise RuntimeError(f"Refusing to reuse incomplete repository directory: {BOOTSTRAP_REPO_ROOT}")
    subprocess.run([
        "git", "clone", "--branch", REPOSITORY_BRANCH, "--single-branch",
        REPOSITORY_URL, str(BOOTSTRAP_REPO_ROOT),
    ], check=True)

actual_branch = subprocess.check_output(
    ["git", "-C", str(BOOTSTRAP_REPO_ROOT), "branch", "--show-current"], text=True
).strip()
if actual_branch != REPOSITORY_BRANCH:
    raise RuntimeError(f"Expected cloned branch {REPOSITORY_BRANCH}, got {actual_branch}")
bootstrap_commit = subprocess.check_output(
    ["git", "-C", str(BOOTSTRAP_REPO_ROOT), "rev-parse", "HEAD"], text=True
).strip()
os.environ["BTXRD_GIT_COMMIT"] = bootstrap_commit
print("Cloned source:", REPOSITORY_URL, actual_branch, bootstrap_commit)

requirements = BOOTSTRAP_REPO_ROOT / "project" / "requirements.txt"
subprocess.run([
    sys.executable, "-m", "pip", "install", "--disable-pip-version-check",
    "--no-input", "-r", str(requirements),
], check=True)
print("Installed pinned dependencies from:", requirements)

dataset_root = Path(os.environ.get("BTXRD_ROOT", str(DEFAULT_DATASET_ROOT))).resolve()
if not (dataset_root / "images").is_dir() or not (dataset_root / "Annotations").is_dir():
    raise FileNotFoundError(
        f"BTXRD is not available at {dataset_root}. Add the Kaggle BTXRD dataset before Run All."
    )
os.environ["BTXRD_ROOT"] = str(dataset_root)

checkpoint_dir = KAGGLE_WORKING_BOOTSTRAP / "checkpoints"
checkpoint_dir.mkdir(parents=True, exist_ok=True)
sam_checkpoint = checkpoint_dir / "sam_vit_b_01ec64.pth"
if not sam_checkpoint.is_file():
    sam_url = "https://dl.fbaipublicfiles.com/segment_anything/sam_vit_b_01ec64.pth"
    temporary_checkpoint = sam_checkpoint.with_suffix(".pth.part")
    print("Downloading SAM ViT-B checkpoint...")
    urllib.request.urlretrieve(sam_url, temporary_checkpoint)
    if temporary_checkpoint.stat().st_size < 300_000_000:
        raise RuntimeError("Downloaded SAM checkpoint is unexpectedly small")
    temporary_checkpoint.replace(sam_checkpoint)
os.environ["SAM_CHECKPOINT"] = str(sam_checkpoint)
print("SAM checkpoint:", sam_checkpoint, "bytes:", sam_checkpoint.stat().st_size)

manifest_override = os.environ.get("BTXRD_SPLIT_MANIFEST", "").strip()
dataset_manifest = dataset_root / "split_manifest.csv"
audit_dir = KAGGLE_WORKING_BOOTSTRAP / "btxrd_split_audit"
generated_manifest = audit_dir / "split_manifest.csv"
if manifest_override:
    split_manifest = Path(manifest_override).resolve()
elif dataset_manifest.is_file():
    split_manifest = dataset_manifest
elif generated_manifest.is_file():
    split_manifest = generated_manifest
else:
    subprocess.run([
        sys.executable, str(BOOTSTRAP_REPO_ROOT / "project" / "tools" / "build_btxrd_split_manifest.py"),
        "--dataset-root", str(dataset_root), "--output-dir", str(audit_dir), "--seed", "42",
    ], cwd=str(BOOTSTRAP_REPO_ROOT), check=True)
    split_manifest = generated_manifest
if not split_manifest.is_file():
    raise FileNotFoundError(split_manifest)
os.environ["BTXRD_SPLIT_MANIFEST"] = str(split_manifest)
print("Split manifest:", split_manifest)


In [ ]:

# Cell 1 - immutable inputs, paths, and run switches
from pathlib import Path
import hashlib
import json
import os
import shlex
import shutil
import subprocess
import sys

NOTEBOOK_ROOT = Path.cwd()
KAGGLE_INPUT = Path("/kaggle/input")
KAGGLE_WORKING = Path("/kaggle/working")
if (NOTEBOOK_ROOT / "project").exists():
    REPO_ROOT = NOTEBOOK_ROOT
else:
    REPO_ROOT = KAGGLE_WORKING / "Thesis"
PROJECT_DIR = REPO_ROOT / "project"

GIT_BRANCH = "pipeline"
EXPECTED_GIT_COMMIT = os.environ.get("BTXRD_GIT_COMMIT", "").strip()
if not EXPECTED_GIT_COMMIT and (REPO_ROOT / ".git").exists():
    EXPECTED_GIT_COMMIT = subprocess.check_output(["git", "-C", str(REPO_ROOT), "rev-parse", "HEAD"], text=True).strip()
if not EXPECTED_GIT_COMMIT:
    raise RuntimeError("Set BTXRD_GIT_COMMIT to the exact committed pipeline revision attached to Kaggle.")

RUN_ID = os.environ.get("BTXRD_RUN_ID", EXPECTED_GIT_COMMIT[:12])
OUTPUT_ROOT = Path(os.environ.get("BTXRD_OUTPUT", str(KAGGLE_WORKING / f"btxrd_thesis_{RUN_ID}")))
DATASET_OVERRIDE = os.environ.get("BTXRD_ROOT", "/kaggle/input/datasets/wanwin/data-btxrd/BTXRD")
SPLIT_MANIFEST_RAW = os.environ.get("BTXRD_SPLIT_MANIFEST", "").strip()
if not SPLIT_MANIFEST_RAW:
    raise RuntimeError("Cell 0 did not provide BTXRD_SPLIT_MANIFEST.")
SPLIT_MANIFEST = Path(SPLIT_MANIFEST_RAW)
SAM_RAW = os.environ.get("SAM_CHECKPOINT", "").strip()
if not SAM_RAW:
    raise RuntimeError("Cell 0 did not provide the downloaded SAM checkpoint.")
SAM_CHECKPOINT = Path(SAM_RAW)

PIPELINE_PROFILE = os.environ.get("BTXRD_PIPELINE_PROFILE", "btxrd_best")
CLASSIFIER_OUTPUT = OUTPUT_ROOT / f"classifier_{PIPELINE_PROFILE}"
PREDICTED_OUTPUT = OUTPUT_ROOT / f"pseudo_predicted_{PIPELINE_PROFILE}"
GROUND_TRUTH_OUTPUT = OUTPUT_ROOT / f"pseudo_ground_truth_{PIPELINE_PROFILE}"
DEBUG_OUTPUT = OUTPUT_ROOT / f"single_image_debug_{PIPELINE_PROFILE}"
EVAL_OUTPUT = OUTPUT_ROOT / f"evaluations_{PIPELINE_PROFILE}"
IMAGE_SIZE = 320
SAM_IMAGE_SIZE = 512
NUM_WORKERS = int(os.environ.get("BTXRD_NUM_WORKERS", "1"))
DEBUG_IMAGE_NAME = os.environ.get("BTXRD_DEBUG_IMAGE", "")

RESUME_UNET_FROM = os.environ.get("BTXRD_RESUME_UNET_FROM", "") or None
RESUME_SUPERVISED_ORACLE_FROM = os.environ.get("BTXRD_RESUME_SUPERVISED_ORACLE_FROM", "") or None
RESUME_UNET_TRAINING_LOG_FROM = os.environ.get("BTXRD_RESUME_UNET_TRAINING_LOG_FROM", "") or None
RESUME_SUPERVISED_ORACLE_TRAINING_LOG_FROM = os.environ.get("BTXRD_RESUME_SUPERVISED_ORACLE_TRAINING_LOG_FROM", "") or None

# Cell 0 bootstraps dependencies and checkpoints before these run switches are evaluated.
INSTALL_DEPENDENCIES = False
RUN_TRAIN_CLASSIFIER = True
RUN_SINGLE_IMAGE_DEBUG = True
RUN_UNET_ONE_EPOCH_PREFLIGHT = True
RUN_FULL_PREDICTED = True
RUN_FULL_GROUND_TRUTH = True
RUN_SUPERVISED_ORACLE_BASELINE = True
RUN_POS_WEIGHT_ABLATION = os.environ.get("BTXRD_RUN_POS_WEIGHT_ABLATION", "0") == "1"
RUN_COMPONENT_TOPK_ABLATION = os.environ.get("BTXRD_RUN_COMPONENT_TOPK_ABLATION", "1") == "1"
RUN_TEST_REPORT = os.environ.get("BTXRD_RUN_LOCKED_TEST", "0") == "1"
FINAL_EVAL_SPLIT = "val"  # Test is inaccessible until Cell 21a freezes and verifies the final config.

print("PROJECT_DIR:", PROJECT_DIR)
print("EXPECTED_GIT_COMMIT:", EXPECTED_GIT_COMMIT)
print("OUTPUT_ROOT (must be new):", OUTPUT_ROOT)
print("SPLIT_MANIFEST:", SPLIT_MANIFEST)
print("SAM_CHECKPOINT:", SAM_CHECKPOINT)
print("FINAL_EVAL_SPLIT:", FINAL_EVAL_SPLIT)


## 1. Repository and environment setup

Kaggle must run with **Internet On**. Cell 0 clones the public `pipeline` branch, records its exact commit, installs the pinned requirements, downloads official SAM ViT-B, and creates the audited split manifest when the dataset does not already provide one. This prevents notebook-only logic from drifting away from `train_classifier.py` and `generate_pseudo_masks.py`.


In [ ]:

# Cell 2 - cloned repository verification and streaming subprocess helper
if not PROJECT_DIR.exists():
    raise FileNotFoundError(
        f"Repository bootstrap did not produce source at {PROJECT_DIR}."
    )
os.chdir(PROJECT_DIR)
GIT_ACTUAL_BRANCH = subprocess.check_output(["git", "-C", str(REPO_ROOT), "branch", "--show-current"], text=True).strip()
GIT_ACTUAL_COMMIT = subprocess.check_output(["git", "-C", str(REPO_ROOT), "rev-parse", "HEAD"], text=True).strip()
GIT_DIRTY_LINES = subprocess.check_output(["git", "-C", str(REPO_ROOT), "status", "--porcelain"], text=True).splitlines()
if GIT_ACTUAL_BRANCH != GIT_BRANCH:
    raise RuntimeError(f"Expected branch {GIT_BRANCH!r}, got {GIT_ACTUAL_BRANCH!r}")
if GIT_ACTUAL_COMMIT != EXPECTED_GIT_COMMIT:
    raise RuntimeError(f"Expected commit {EXPECTED_GIT_COMMIT}, got {GIT_ACTUAL_COMMIT}")
if GIT_DIRTY_LINES:
    raise RuntimeError(f"Final notebook refuses a dirty tree: {GIT_DIRTY_LINES[:10]}")
print("Verified clean source:", GIT_ACTUAL_BRANCH, GIT_ACTUAL_COMMIT)
if str(PROJECT_DIR) not in sys.path:
    sys.path.insert(0, str(PROJECT_DIR))

def run_streaming(cmd, cwd=PROJECT_DIR, check=True):
    cmd = [str(value) for value in cmd]
    print("$ " + " ".join(shlex.quote(value) for value in cmd), flush=True)
    process = subprocess.Popen(cmd, cwd=str(cwd), stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, encoding="utf-8", errors="replace")
    for line in process.stdout:
        print(line.rstrip("\n"), flush=True)
    code = process.wait()
    if check and code:
        raise subprocess.CalledProcessError(code, cmd)
    return code

if INSTALL_DEPENDENCIES:
    raise RuntimeError("Dependencies are already installed in Cell 0; do not reinstall them mid-run.")
print("cwd:", Path.cwd())


In [ ]:

# Cell 3 - hardware/runtime audit (fail-fast)
import numpy as np
import pandas as pd
from PIL import Image
import matplotlib.pyplot as plt
import torch

print("Python:", sys.version)
print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
print("CUDA devices:", torch.cuda.device_count())
if not torch.cuda.is_available():
    raise RuntimeError("CUDA is required for the full classifier + SAM + U-Net pipeline.")
for index in range(torch.cuda.device_count()):
    props = torch.cuda.get_device_properties(index)
    print(index, torch.cuda.get_device_name(index), round(props.total_memory / 2**30, 2), "GiB")
free_gb = shutil.disk_usage(KAGGLE_WORKING if KAGGLE_WORKING.exists() else NOTEBOOK_ROOT).free / 2**30
print("Free disk GiB:", round(free_gb, 2))
if free_gb < 25:
    raise RuntimeError("At least 25 GiB free disk is required for staged checkpoints/masks/reports.")


## 2. Dataset resolution, split audit, and leakage guard

BTXRD has no predefined split. The notebook builds an immutable derived manifest with exact-hash deduplication and a documented heuristic case grouping.
The grouping is not patient/lesion verified because the release has no such identifier; the manifest and this limitation are carried into every stage.


In [ ]:

# Cell 4 - locate immutable BTXRD inputs and run strict Kaggle preflight
from datasets.btxrd import TUMOR_TYPE_CLASS_NAMES, load_btxrd_records, resolve_btxrd_root, split_btxrd_records
from datasets.factory import build_classification_dataset, build_segmentation_dataset

def find_btxrd_root(base: Path):
    candidates = [base, *base.glob("*"), *base.glob("*/*")] if base and base.exists() else []
    for candidate in candidates:
        try:
            return resolve_btxrd_root(candidate)
        except FileNotFoundError:
            pass
    return None

search_root = Path(DATASET_OVERRIDE) if DATASET_OVERRIDE else KAGGLE_INPUT
BTXRD_ROOT = find_btxrd_root(search_root)
if BTXRD_ROOT is None:
    raise FileNotFoundError("BTXRD not found. Set BTXRD_ROOT or attach images/, Annotations/, dataset.csv/xlsx.")
for required in (SPLIT_MANIFEST, SAM_CHECKPOINT):
    if not required.is_file():
        raise FileNotFoundError(required)
if OUTPUT_ROOT.exists():
    raise FileExistsError(f"Refusing to reuse existing run directory: {OUTPUT_ROOT}. Choose a new BTXRD_RUN_ID.")

READINESS_REPORT = OUTPUT_ROOT.parent / f"{OUTPUT_ROOT.name}_preflight.json"
run_streaming([
    sys.executable, "tools/validate_kaggle_readiness.py",
    "--dataset-root", str(BTXRD_ROOT), "--split-manifest", str(SPLIT_MANIFEST),
    "--sam-checkpoint", str(SAM_CHECKPOINT), "--output-root", str(OUTPUT_ROOT),
    "--expected-commit", EXPECTED_GIT_COMMIT, "--report-json", str(READINESS_REPORT),
])
for directory in (OUTPUT_ROOT, CLASSIFIER_OUTPUT, PREDICTED_OUTPUT, GROUND_TRUTH_OUTPUT, DEBUG_OUTPUT, EVAL_OUTPUT):
    directory.mkdir(parents=True, exist_ok=False)
records = load_btxrd_records(BTXRD_ROOT, split_manifest=SPLIT_MANIFEST)
print("BTXRD_ROOT:", BTXRD_ROOT)
print("records:", len(records))


In [ ]:

# Cell 5 — split/class distribution and deterministic 5-image smoke list
def split_frame(name):
    rows = split_btxrd_records(records, split=name, seed=42)
    frame = pd.DataFrame(rows)
    frame["tumor_type_name"] = frame["tumor_type"].map(dict(enumerate(TUMOR_TYPE_CLASS_NAMES)))
    return frame

split_frames = {name: split_frame(name) for name in ["train", "val"]}
parts = []
for name, frame in split_frames.items():
    counts = frame["tumor_type_name"].value_counts().reindex(TUMOR_TYPE_CLASS_NAMES, fill_value=0)
    parts.append(pd.DataFrame({"split": name, "class": counts.index, "count": counts.values}))
    print(f"{name}: n={len(frame)} tumor={int((frame.tumor_type > 0).sum())} normal={int((frame.tumor_type == 0).sum())}")
split_summary = pd.concat(parts, ignore_index=True)
display(split_summary.pivot(index="class", columns="split", values="count"))
split_summary.pivot(index="class", columns="split", values="count").plot.bar(figsize=(14, 4), grid=True)
plt.title("BTXRD development splits by tumor_type (test remains sealed)"); plt.tight_layout(); plt.show()

if not DEBUG_IMAGE_NAME:
    DEBUG_IMAGE_NAME = str(split_frames["val"].query("tumor_type > 0").iloc[0]["image_id"])
smoke_pool = (split_frames["val"].sort_values("image_id").groupby("tumor_type", sort=True).head(1))
DEBUG_SMOKE_IMAGES = [DEBUG_IMAGE_NAME] + [str(x) for x in smoke_pool.image_id if str(x) != DEBUG_IMAGE_NAME]
DEBUG_SMOKE_IMAGES = DEBUG_SMOKE_IMAGES[:5]
assert 3 <= len(DEBUG_SMOKE_IMAGES) <= 5
DEBUG_IMAGE_LIST = OUTPUT_ROOT / "debug_image_list.txt"
DEBUG_IMAGE_LIST.write_text("\n".join(DEBUG_SMOKE_IMAGES) + "\n", encoding="utf-8")
print("DEBUG_IMAGE_NAME:", DEBUG_IMAGE_NAME)
print("GPU prompt-ensemble smoke images:", DEBUG_SMOKE_IMAGES)


In [ ]:

# Cell 6 — leakage guard: classification labels vs polygon masks
classification_ds = build_classification_dataset(
    "btxrd", root=BTXRD_ROOT, split="val", target_columns=["tumor_type"], image_size=IMAGE_SIZE,
    split_manifest=SPLIT_MANIFEST,
)
segmentation_ds = build_segmentation_dataset("btxrd", root=BTXRD_ROOT, split="val", image_size=IMAGE_SIZE, augment=False, split_manifest=SPLIT_MANIFEST)
classification_names = {str(x["image_id"]) for x in classification_ds.samples}
segmentation_names = {str(x["image_id"]) for x in segmentation_ds.samples}
print("classification records:", len(classification_names))
print("segmentation records:", len(segmentation_names))
print("same IDs:", classification_names == segmentation_names)
assert classification_ds.target_columns == ["tumor_type"]
print("Generation will use image-level target only:", classification_ds.target_columns)
print("Segmentation masks remain diagnostic/evaluation-only objects.")


## 3. Ground-truth visualization (diagnostic only)

This cell rasterizes LabelMe polygons so annotation errors can be seen. It must not be used as an input to any
generation cell. The final plot title explicitly marks the polygon as diagnostic.


In [ ]:

# Cell 7 — original image, polygon GT, and overlay
def show_gt_case(want_tumor: bool):
    frame = split_frames["val"]
    row = frame[frame.tumor_type.gt(0) if want_tumor else frame.tumor_type.eq(0)].iloc[0]
    image_name = str(row.image_id)
    image = Image.open(BTXRD_ROOT / "images" / image_name).convert("RGB").resize((IMAGE_SIZE, IMAGE_SIZE))
    index = next(i for i, x in enumerate(segmentation_ds.samples) if str(x["image_id"]) == image_name)
    _, mask_tensor, _ = segmentation_ds[index]
    mask = mask_tensor[0].numpy() > 0.5
    image_np = np.asarray(image)
    overlay = image_np.copy()
    overlay[mask] = (0.45 * overlay[mask] + 0.55 * np.array([255, 30, 30])).astype(np.uint8)
    fig, axes = plt.subplots(1, 3, figsize=(13, 4))
    axes[0].imshow(image_np); axes[0].set_title(f"{image_name}\noriginal")
    axes[1].imshow(mask, cmap="gray"); axes[1].set_title("polygon GT")
    axes[2].imshow(overlay); axes[2].set_title("diagnostic overlay")
    for ax in axes: ax.axis("off")
    plt.tight_layout(); plt.show()

show_gt_case(True)
show_gt_case(False)


## 4. SAM checkpoint and canonical classifier training

The next cells train the paired image-level classifier using `--pipeline-profile btxrd_best`. The profile fixes
`tumor_type`, 320 px, batch 4, up to 30 epochs with early-stop patience 7, seed 42, inverse-frequency CE, and disables PuzzleCAM/teacher-attention
losses for the selected CE/320 model.


In [ ]:

# Cell 8 - verify the SAM checkpoint downloaded by Cell 0
if not SAM_CHECKPOINT.is_file():
    raise FileNotFoundError("Cell 0 did not download sam_vit_b_01ec64.pth.")
print("SAM:", SAM_CHECKPOINT, "GiB:", round(SAM_CHECKPOINT.stat().st_size / 2**30, 3),
      "sha256:", hashlib.sha256(SAM_CHECKPOINT.read_bytes()).hexdigest())

# Cell 9 - train classifier
CLASSIFIER_CHECKPOINT = CLASSIFIER_OUTPUT / "best_classifier.pt"
_cam_preview_epochs = "1,5,10,20,30"
classifier_cmd = [
    sys.executable, "train_classifier.py", "--pipeline-profile", PIPELINE_PROFILE,
    "--data-root", str(BTXRD_ROOT), "--split-manifest", str(SPLIT_MANIFEST), "--num-workers", str(NUM_WORKERS),
    "--epochs", "30", "--early-stop-patience", "7", "--save-cam-epochs", _cam_preview_epochs, "--cam-preview-count", "4", "--output-dir", str(CLASSIFIER_OUTPUT),
]
if RUN_TRAIN_CLASSIFIER:
    run_streaming(classifier_cmd)
if not CLASSIFIER_CHECKPOINT.exists():
    raise FileNotFoundError(CLASSIFIER_CHECKPOINT)
CLASSIFIER_BUDGET_AUDIT = CLASSIFIER_OUTPUT / "classifier_epoch_budget_audit.json"
if not CLASSIFIER_BUDGET_AUDIT.exists():
    raise FileNotFoundError(CLASSIFIER_BUDGET_AUDIT)
budget_audit = json.loads(CLASSIFIER_BUDGET_AUDIT.read_text(encoding="utf-8"))
print(json.dumps(budget_audit, indent=2))
if budget_audit["assessment"] != "plateau_or_decline_observed":
    raise RuntimeError("Classifier epoch budget is not supported by the audited validation curve. Extend the otherwise-identical budget (for example BTXRD_PIPELINE_PROFILE=btxrd_hybrid), rerun validation, and do not freeze test yet.")


In [ ]:

# Cell 10 — training curves and checkpoint audit
training_log = CLASSIFIER_OUTPUT / "training_log.csv"
if training_log.exists():
    train_df = pd.read_csv(training_log)
    display(train_df)
    fig, axes = plt.subplots(1, 3, figsize=(17, 4))
    train_df[["train_loss", "val_loss"]].plot(ax=axes[0], marker="o", title="CE loss")
    train_df[["train_acc", "val_acc"]].plot(ax=axes[1], marker="o", title="accuracy")
    train_df[["train_f1", "val_f1"]].plot(ax=axes[2], marker="o", title="macro-F1")
    for ax in axes: ax.grid(alpha=0.3)
    plt.tight_layout(); plt.show()

state = torch.load(CLASSIFIER_CHECKPOINT, map_location="cpu")
expected = {"task": "single-label", "target_columns": ["tumor_type"], "num_classes": 10, "normalization": "imagenet"}
for key, value in expected.items():
    print(key, state.get(key), "expected", value)
    assert state.get(key) == value
print("checkpoint:", CLASSIFIER_CHECKPOINT)


# Complete classifier and tumor-vs-normal gate report on validation only.
CLASSIFIER_EVAL_OUTPUT = EVAL_OUTPUT / "classifier_val"
run_streaming([
    sys.executable, "evaluate_classifier.py", "--data-root", str(BTXRD_ROOT),
    "--split", "val", "--split-manifest", str(SPLIT_MANIFEST), "--checkpoint", str(CLASSIFIER_CHECKPOINT),
    "--image-size", str(IMAGE_SIZE), "--num-workers", str(NUM_WORKERS), "--output-dir", str(CLASSIFIER_EVAL_OUTPUT),
])
display(pd.json_normalize(json.loads((CLASSIFIER_EVAL_OUTPUT / "summary.json").read_text())).T)


In [ ]:
# Cell 10a - mandatory one-epoch U-Net execution preflight before any full pseudo-mask run
# This uses train polygons only as an execution smoke test; its checkpoint is never a WSSS result.
UNET_PREFLIGHT_OUTPUT = OUTPUT_ROOT / f"unet_one_epoch_preflight_{PIPELINE_PROFILE}"
if RUN_UNET_ONE_EPOCH_PREFLIGHT:
    run_streaming([
        sys.executable, "train_segmentation.py", "--pipeline-profile", PIPELINE_PROFILE,
        "--data-root", str(BTXRD_ROOT), "--split-manifest", str(SPLIT_MANIFEST),
        "--train-split", "train", "--val-split", "val", "--image-size", str(IMAGE_SIZE),
        "--batch-size", "2", "--num-workers", str(NUM_WORKERS), "--epochs", "1",
        "--pos-weight-mode", "auto-clamped", "--output-dir", str(UNET_PREFLIGHT_OUTPUT),
    ])
    preflight_checkpoint = UNET_PREFLIGHT_OUTPUT / "best_unet.pt"
    preflight_log = UNET_PREFLIGHT_OUTPUT / "training_log.csv"
    if not preflight_checkpoint.is_file() or len(pd.read_csv(preflight_log)) != 1:
        raise RuntimeError("U-Net preflight did not complete exactly one epoch; stop before full pseudo-mask generation.")
    print("U-Net one-epoch preflight passed:", preflight_checkpoint)
else:
    raise RuntimeError("RUN_UNET_ONE_EPOCH_PREFLIGHT must remain enabled for a fresh final run.")


In [ ]:

# Cell 11 — CAM snapshots across epochs
from collections import defaultdict
import re
cam_dir = CLASSIFIER_OUTPUT / "cam_preview"
by_sample = defaultdict(list)
for path in sorted(cam_dir.glob("cam_epoch*.png")) if cam_dir.exists() else []:
    match = re.match(r"cam_epoch(\d+)_(.+)\.png", path.name)
    if match: by_sample[match.group(2)].append((int(match.group(1)), path))
if by_sample:
    rows = sorted(by_sample.items()); ncols = max(len(x) for _, x in rows)
    fig, axes = plt.subplots(len(rows), ncols, figsize=(3.2*ncols, 3.2*len(rows)), squeeze=False)
    for r, (stem, entries) in enumerate(rows):
        for c, (epoch, path) in enumerate(sorted(entries)):
            axes[r, c].imshow(Image.open(path)); axes[r, c].set_title(f"{stem}\nepoch {epoch}"); axes[r, c].axis("off")
        for c in range(len(entries), ncols): axes[r, c].axis("off")
    plt.tight_layout(); plt.show()
else:
    print("No CAM snapshots found.")


## 5. Single-image pre-SAM trace

This trace stops before SAM and exposes the exact class choice, class-vs-normal CAM, percentile supports, connected
components, box padding, and positive/negative points. The `predicted` trace is end-to-end behavior; the optional
`ground_truth` trace is a localization diagnostic and is never passed to pseudo-mask generation unless explicitly run
as the separate oracle protocol below.


In [ ]:

# Cell 12 - LayerCAM, threshold, components, boxes, and points for one image
from models.layercam import LayerCAM
from pseudo.generate_layercam import generate_fused_cam
from pseudo.tumor_morphology import build_class_conditioned_components
from generate_pseudo_masks import load_classifier

classifier, checkpoint_task, checkpoint_normalization = load_classifier(
    CLASSIFIER_CHECKPOINT, fallback_num_classes=10, device=torch.device("cpu"), expected_target_columns=["tumor_type"],
    expected_task="single-label", expected_num_classes=10,
)
classifier.eval()
trace_ds = build_classification_dataset(
    "btxrd", root=BTXRD_ROOT, split="val", target_columns=["tumor_type"], image_size=IMAGE_SIZE,
    split_manifest=SPLIT_MANIFEST,
)
trace_index = next(i for i, x in enumerate(trace_ds.samples) if str(x["image_id"]) == DEBUG_IMAGE_NAME)
trace_tensor, trace_target, _ = trace_ds[trace_index]
trace_rgb = np.asarray(Image.open(BTXRD_ROOT / "images" / DEBUG_IMAGE_NAME).convert("RGB").resize((IMAGE_SIZE, IMAGE_SIZE)))

def _norm_map(x):
    x = np.asarray(x, dtype=np.float32)
    return (x - x.min()) / (x.max() - x.min() + 1e-8)

def trace_pre_sam(image_name=DEBUG_IMAGE_NAME, protocol="predicted"):
    idx = next(i for i, x in enumerate(trace_ds.samples) if str(x["image_id"]) == str(image_name))
    image_tensor, target, _ = trace_ds[idx]
    batch = image_tensor.unsqueeze(0)
    with torch.no_grad():
        logits = classifier(batch)
        probs = torch.softmax(logits, dim=1)[0].cpu().numpy()
    selected = int(probs.argmax()) if protocol == "predicted" else int(target)
    production_skipped = protocol == "predicted" and selected == 0
    if selected == 0:
        # A forced tumor-class CAM is diagnostic only; production writes an empty mask.
        selected = int(np.argsort(probs[1:])[-1] + 1)
    weights = np.zeros(10, dtype=np.float32); weights[selected] = 1.0
    cam_engine = LayerCAM(
        classifier, device=torch.device("cpu"), layer_weights=[0.2, 0.3, 0.5], gradient_mode="positive",
    )
    fused_cam, _, _ = generate_fused_cam(cam_engine, batch, class_weights=weights, confidence_threshold=0.5)
    contrast_output = cam_engine.cam_for_class_contrast(batch, selected, reference_index=0)
    contrast = _norm_map(contrast_output.cam.detach().cpu().numpy()[0])
    components = {}
    for percentile in [85, 90, 95]:
        likelihood, support, component_list = build_class_conditioned_components(
            trace_rgb, [contrast], [1.0], cam_percentile=percentile,
            min_component_area=100, max_components=3, points_per_component=5,
            bbox_padding_ratio=0.02, negative_points_per_component=4,
        )
        components[percentile] = {"likelihood": likelihood, "support_mask": support, "components": component_list}
    return {"image": trace_rgb, "target": int(target), "selected": selected, "probs": probs,
            "production_skipped": production_skipped,
            "cam": fused_cam, "contrast": contrast, "components": components}

trace_predicted = trace_pre_sam(protocol="predicted")
trace_ground_truth = trace_pre_sam(protocol="ground_truth")
print("image:", DEBUG_IMAGE_NAME, "image-level target:", trace_predicted["target"],
      "diagnostic CAM class:", trace_predicted["selected"], "GT class:", trace_ground_truth["selected"],
      "production skipped predicted-normal:", trace_predicted["production_skipped"])

def plot_pre_sam_trace(trace, title):
    fig, axes = plt.subplots(1, 5, figsize=(21, 4))
    axes[0].imshow(trace["image"]); axes[0].set_title(f"Original\n{title}")
    axes[1].imshow(trace["image"]); axes[1].imshow(trace["cam"], cmap="magma", alpha=.48); axes[1].set_title("Fused LayerCAM")
    axes[2].imshow(trace["image"]); axes[2].imshow(trace["contrast"], cmap="jet", alpha=.48); axes[2].set_title("Class-vs-normal CAM")
    support = trace["components"][85]["support_mask"]
    axes[3].imshow(trace["image"]); axes[3].imshow(support, cmap="Greens", alpha=.45); axes[3].set_title("p85 support")
    axes[4].imshow(trace["image"])
    for comp in trace["components"][85]["components"]:
        x0, y0, x1, y1 = comp.bbox
        axes[4].add_patch(plt.Rectangle((x0, y0), x1-x0, y1-y0, fill=False, color="yellow", linewidth=2))
        pos = np.asarray(comp.positive_points); neg = np.asarray(comp.negative_points)
        if len(pos): axes[4].scatter(pos[:, 0], pos[:, 1], c="red", s=25, label="positive")
        if len(neg): axes[4].scatter(neg[:, 0], neg[:, 1], c="cyan", s=20, label="negative")
    axes[4].set_title("Box + points")
    for ax in axes: ax.axis("off")
    plt.tight_layout(); plt.show()

plot_pre_sam_trace(trace_predicted, "predicted protocol" if not trace_predicted["production_skipped"]
                   else "forced-non-normal diagnostic (production skips)")
plot_pre_sam_trace(trace_ground_truth, "ground-truth class (diagnostic)")
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, percentile in zip(axes, [85, 90, 95]):
    ax.imshow(trace_predicted["image"])
    ax.imshow(trace_predicted["components"][percentile]["support_mask"], cmap="Greens", alpha=.45)
    ax.set_title(f"predicted support p{percentile}\\ncomponents={len(trace_predicted['components'][percentile]['components'])}")
    ax.axis("off")
plt.tight_layout(); plt.show()


## 6. Actual SAM candidate trace (one image)

The next cell invokes the production generator with `--debug`, so the displayed candidates are not a notebook
reimplementation. It records every SAM mask and score before selection, plus morphology intermediates.


In [ ]:

# Cell 13 - GPU smoke test of the full prompt ensemble on 3–5 deterministic validation images
DEBUG_OUTPUT.mkdir(parents=True, exist_ok=True)
debug_cmd = [
    sys.executable, "generate_pseudo_masks.py", "--pipeline-profile", PIPELINE_PROFILE,
    "--data-root", str(BTXRD_ROOT), "--split-manifest", str(SPLIT_MANIFEST), "--split", "val", "--classifier-checkpoint", str(CLASSIFIER_CHECKPOINT),
    "--sam-checkpoint", str(SAM_CHECKPOINT), "--image-list", str(DEBUG_IMAGE_LIST), "--max-images", str(len(DEBUG_SMOKE_IMAGES)),
    "--debug", "--evaluate-prompt-quality", "--save-visuals-limit", str(len(DEBUG_SMOKE_IMAGES)), "--output-dir", str(DEBUG_OUTPUT),
]
if RUN_SINGLE_IMAGE_DEBUG:
    if not torch.cuda.is_available():
        raise RuntimeError("Canonical SAM smoke test requires a CUDA GPU; run this cell on Kaggle GPU.")
    run_streaming(debug_cmd)
else:
    print("RUN_SINGLE_IMAGE_DEBUG=False; command prepared only.")


In [ ]:

# Cell 14 - display actual CAM/morphology/SAM artifacts and candidate table
debug_case_dir = DEBUG_OUTPUT / "debug" / Path(DEBUG_IMAGE_NAME).stem
def display_artifact(path, title=None, cmap=None):
    if not path.exists():
        print("missing:", path); return
    image = Image.open(path)
    plt.figure(figsize=(4, 4)); plt.imshow(image, cmap=cmap); plt.title(title or path.name); plt.axis("off"); plt.show()

for filename in ["simple_tumor_likelihood.png", "simple_tumor_support.png", "tumor_likelihood.png",
                 "tumor_seeds.png", "tumor_support.png"]:
    display_artifact(debug_case_dir / filename)
display_artifact(DEBUG_OUTPUT / "masks" / f"{Path(DEBUG_IMAGE_NAME).stem}.png", "final pseudo-mask", cmap="gray")
score_path = debug_case_dir / "scores.json"
if score_path.exists():
    score_records = json.loads(score_path.read_text(encoding="utf-8"))
    candidate_table = pd.DataFrame.from_dict(score_records, orient="index").rename_axis("candidate").reset_index()
    display(candidate_table.head(12))
    overlay_paths = sorted(debug_case_dir.glob("overlay_mask_*.png"))[:12]
    if overlay_paths:
        fig, axes = plt.subplots(3, 4, figsize=(16, 12)); axes = axes.ravel()
        for ax, path in zip(axes, overlay_paths):
            ax.imshow(Image.open(path)); ax.set_title(path.name); ax.axis("off")
        for ax in axes[len(overlay_paths):]: ax.axis("off")
        plt.tight_layout(); plt.show()
else:
    print("No scores.json yet; run Cell 13 first.")


## 7. End-to-end predicted protocol

`CAM_TARGET_CLASS=predicted` is the inference-realistic diagnostic for the pseudo-label generator: its classifier
prediction, not the image-level annotation, chooses the CAM class. It is not the final deployed path, which is
U-Net-only after WSSS training. Results are saved under a dedicated directory and are never merged with the
known-image-label localization protocol.


In [ ]:

# Cell 15 - full validation pseudo-masks: predicted protocol
PREDICTED_CMD = [
    sys.executable, "generate_pseudo_masks.py", "--pipeline-profile", PIPELINE_PROFILE,
    "--data-root", str(BTXRD_ROOT), "--split-manifest", str(SPLIT_MANIFEST), "--split", "val", "--classifier-checkpoint", str(CLASSIFIER_CHECKPOINT),
    "--sam-checkpoint", str(SAM_CHECKPOINT), "--process-all", "--evaluate-prompt-quality",
    "--save-visuals-limit", "10", "--output-dir", str(PREDICTED_OUTPUT),
]
if RUN_FULL_PREDICTED:
    run_streaming(PREDICTED_CMD)
else:
    print("RUN_FULL_PREDICTED=False; command prepared only.")


In [ ]:

# Cell 16 - evaluate predicted protocol against polygons (evaluation only)
PREDICTED_EVAL = EVAL_OUTPUT / "predicted.csv"
PREDICTED_EVAL_JSON = EVAL_OUTPUT / "predicted.json"
predicted_eval_cmd = [
    sys.executable, "evaluate_pseudo_masks.py", "--data-root", str(BTXRD_ROOT),
    "--split-manifest", str(SPLIT_MANIFEST),
    "--split", "val", "--image-size", str(IMAGE_SIZE), "--pred-mask-root", str(PREDICTED_OUTPUT / "masks"),
    "--output-csv", str(PREDICTED_EVAL), "--output-json", str(PREDICTED_EVAL_JSON),
]
if RUN_FULL_PREDICTED:
    run_streaming(predicted_eval_cmd)
else:
    print("Evaluation command prepared:", predicted_eval_cmd)


## 8. Localization/oracle protocol (kept separate)

`CAM_TARGET_CLASS=ground_truth` supplies the known image-level class only to test localization quality. It is not an
end-to-end result and must not be compared as if it were deployment inference. Polygon masks remain evaluation-only.


In [ ]:

# Cell 17 - full validation pseudo-masks: ground-truth-class localization protocol
GROUND_TRUTH_CMD = [
    sys.executable, "generate_pseudo_masks.py", "--pipeline-profile", PIPELINE_PROFILE,
    "--data-root", str(BTXRD_ROOT), "--split-manifest", str(SPLIT_MANIFEST), "--split", "val", "--classifier-checkpoint", str(CLASSIFIER_CHECKPOINT),
    "--sam-checkpoint", str(SAM_CHECKPOINT), "--process-all", "--cam-target-class", "ground_truth",
    "--evaluate-prompt-quality", "--save-visuals-limit", "10", "--output-dir", str(GROUND_TRUTH_OUTPUT),
]
if RUN_FULL_GROUND_TRUTH:
    run_streaming(GROUND_TRUTH_CMD)
else:
    print("RUN_FULL_GROUND_TRUTH=False; command prepared only.")


In [ ]:

# Cell 18 - evaluate ground-truth-class protocol separately
GROUND_TRUTH_EVAL = EVAL_OUTPUT / "ground_truth.csv"
GROUND_TRUTH_EVAL_JSON = EVAL_OUTPUT / "ground_truth.json"
ground_truth_eval_cmd = [
    sys.executable, "evaluate_pseudo_masks.py", "--data-root", str(BTXRD_ROOT),
    "--split-manifest", str(SPLIT_MANIFEST),
    "--split", "val", "--image-size", str(IMAGE_SIZE), "--pred-mask-root", str(GROUND_TRUTH_OUTPUT / "masks"),
    "--output-csv", str(GROUND_TRUTH_EVAL), "--output-json", str(GROUND_TRUTH_EVAL_JSON),
]
if RUN_FULL_GROUND_TRUTH:
    run_streaming(ground_truth_eval_cmd)
else:
    print("Evaluation command prepared:", ground_truth_eval_cmd)


In [ ]:
# Cell 18a - validation-only component_topk ablation driven by GT component statistics
# Polygon GT is used only to choose/report ablation values and evaluate outputs; it never enters CAM/SAM generation.
if RUN_COMPONENT_TOPK_ABLATION:
    gt_component_summary = json.loads(GROUND_TRUTH_EVAL_JSON.read_text(encoding="utf-8"))
    component_histogram = {int(k): int(v) for k, v in gt_component_summary["gt_component_count_histogram"].items()}
    observed_max_components = max(component_histogram, default=1)
    topk_values = list(range(1, min(3, observed_max_components) + 1))
    print("GT component histogram:", component_histogram, "ablation top-k:", topk_values)
    topk_rows = []
    for topk in topk_values:
        if topk == 3:
            summary_path = GROUND_TRUTH_EVAL_JSON
        else:
            ablation_output = OUTPUT_ROOT / f"component_topk_{topk}_{PIPELINE_PROFILE}"
            summary_path = EVAL_OUTPUT / f"component_topk_{topk}.json"
            run_streaming([
                sys.executable, "generate_pseudo_masks.py", "--pipeline-profile", PIPELINE_PROFILE,
                "--allow-validation-component-topk-ablation", "--component-topk", str(topk),
                "--data-root", str(BTXRD_ROOT), "--split-manifest", str(SPLIT_MANIFEST), "--split", "val",
                "--classifier-checkpoint", str(CLASSIFIER_CHECKPOINT), "--sam-checkpoint", str(SAM_CHECKPOINT),
                "--cam-target-class", "ground_truth", "--process-all", "--save-visuals-limit", "0",
                "--output-dir", str(ablation_output),
            ])
            run_streaming([
                sys.executable, "evaluate_pseudo_masks.py", "--data-root", str(BTXRD_ROOT),
                "--split-manifest", str(SPLIT_MANIFEST), "--split", "val", "--image-size", str(IMAGE_SIZE),
                "--pred-mask-root", str(ablation_output / "masks"),
                "--output-csv", str(EVAL_OUTPUT / f"component_topk_{topk}.csv"), "--output-json", str(summary_path),
            ])
        summary = json.loads(summary_path.read_text(encoding="utf-8"))
        topk_rows.append({"component_topk": topk, **{k: summary.get(k) for k in ["mean_tumor_dice", "lesion_one_to_one_iou10_recall", "lesion_one_to_one_iou25_recall", "lesion_one_to_one_iou50_recall"]}})
    display(pd.DataFrame(topk_rows))
else:
    print("Set BTXRD_RUN_COMPONENT_TOPK_ABLATION=1 for the validation-only top-k ablation.")


## 9. Metrics and oracle diagnostics

The evaluation CSVs are summarized side by side. The prompt-quality file decomposes errors into CAM/support loss,
SAM candidate loss, selection loss, and post-processing delta. This is the main debugging table; macro-F1 is not a
localization metric and is intentionally not substituted for Dice.


In [ ]:

# Cell 19 - protocols remain separate; summary JSON is never mixed into per-image CSV
def flatten_protocol_summary(path, protocol):
    if not path.exists():
        return {"protocol": protocol, "status": "not run"}
    payload = json.loads(path.read_text())
    row = {"protocol": protocol, "status": "complete", "cam_target_class_protocol": payload.get("cam_target_class_protocol")}
    for section in ("end_to_end", "cam_sam_conditional_tumor_only"):
        for key, value in (payload.get(section) or {}).items():
            row[f"{section}.{key}"] = value
    return row

protocol_summary = pd.DataFrame([
    flatten_protocol_summary(PREDICTED_EVAL_JSON, "predicted"),
    flatten_protocol_summary(GROUND_TRUTH_EVAL_JSON, "ground_truth"),
])
display(protocol_summary.T)

def load_quality(output_dir, protocol):
    path = output_dir / "prompt_quality.csv"
    if not path.exists(): return pd.DataFrame()
    frame = pd.read_csv(path); frame.insert(0, "protocol", protocol); return frame

quality = pd.concat([load_quality(PREDICTED_OUTPUT, "predicted"), load_quality(GROUND_TRUTH_OUTPUT, "ground_truth")], ignore_index=True)
if not quality.empty:
    diagnostic_columns = ["protocol", "tumor_type", "foreground_iou", "foreground_recall", "foreground_precision",
                          "point_hit_rate", "box_recall", "box_precision", "oracle_best_single_dice",
                          "oracle_best_single_dice_clipped", "selected_dice", "support_loss_dice",
                          "selection_loss_dice", "final_dice", "postprocess_delta_dice"]
    display(quality[diagnostic_columns].describe(include="all").T)
    display(quality.groupby(["protocol", "tumor_type"])[["selected_dice", "final_dice", "support_loss_dice", "selection_loss_dice"]].mean())
else:
    print("No prompt_quality.csv found.")


In [ ]:

# Cell 20 - qualitative predicted-vs-localization panel (polygon is diagnostic only)
def find_visual(output_dir, image_name):
    candidates = [output_dir / "overlays" / f"{Path(image_name).stem}_fused_layercam.png",
                  output_dir / "masks" / f"{Path(image_name).stem}.png",
                  output_dir / "debug" / Path(image_name).stem / "overlay_mask_0.png"]
    return next((p for p in candidates if p.exists()), None)

pred_visual = find_visual(PREDICTED_OUTPUT, DEBUG_IMAGE_NAME)
gt_visual = find_visual(GROUND_TRUTH_OUTPUT, DEBUG_IMAGE_NAME)
gt_image = Image.open(BTXRD_ROOT / "images" / DEBUG_IMAGE_NAME).convert("RGB").resize((IMAGE_SIZE, IMAGE_SIZE))
fig, axes = plt.subplots(1, 4, figsize=(17, 4))
axes[0].imshow(gt_image); axes[0].set_title("Original")
axes[1].imshow(gt_image); axes[1].imshow(trace_predicted["contrast"], cmap="jet", alpha=.45); axes[1].set_title("Predicted-class CAM")
if pred_visual: axes[2].imshow(Image.open(pred_visual)); axes[2].set_title("Predicted protocol output")
else: axes[2].text(.5, .5, "predicted visual missing", ha="center"); axes[2].set_title("Predicted protocol")
if gt_visual: axes[3].imshow(Image.open(gt_visual)); axes[3].set_title("GT-class diagnostic output")
else: axes[3].text(.5, .5, "ground_truth visual missing", ha="center"); axes[3].set_title("GT-class protocol")
for ax in axes: ax.axis("off")
plt.tight_layout(); plt.show()


## 9b. Train split pseudo-masks + U-Net trained on the pipeline's own output

Sections 7-8 only generated pseudo-masks for `val` (enough for the oracle diagnostics above). Training a
U-Net on the canonical WSSS output requires pseudo-masks for `train` too. The next cell generates
those using the known image-level label (ground-truth target protocol), then supplies only the training
pseudo-masks via `--train-pred-mask-root`. Held-out validation polygons are used solely for model selection;
this measures the pipeline's true end-to-end segmentation quality, as opposed to section 10's
supervised-oracle baseline (GT-trained, upper bound only).


In [ ]:
# Generate canonical WSSS pseudo-masks for the train split. The classifier
# is trained with image-level labels, so the known label is used to select
# the CAM target; predicted-class train masks remain a separate diagnostic.
TRAIN_GROUND_TRUTH_OUTPUT = OUTPUT_ROOT / f"pseudo_ground_truth_train_{PIPELINE_PROFILE}"
RUN_FULL_GROUND_TRUTH_TRAIN = True

TRAIN_GROUND_TRUTH_CMD = [
    sys.executable, "generate_pseudo_masks.py", "--pipeline-profile", PIPELINE_PROFILE,
    "--data-root", str(BTXRD_ROOT), "--split-manifest", str(SPLIT_MANIFEST), "--split", "train", "--classifier-checkpoint", str(CLASSIFIER_CHECKPOINT),
    "--cam-target-class", "ground_truth",
    "--sam-checkpoint", str(SAM_CHECKPOINT), "--process-all",
    "--save-visuals-limit", "0", "--output-dir", str(TRAIN_GROUND_TRUTH_OUTPUT),
]
print(' '.join(TRAIN_GROUND_TRUTH_CMD))
if RUN_FULL_GROUND_TRUTH_TRAIN:
    run_streaming(TRAIN_GROUND_TRUTH_CMD)
else:
    print("RUN_FULL_GROUND_TRUTH_TRAIN=False; command prepared only.")


In [ ]:
# Train U-Net on canonical train-split WSSS pseudo-masks (known image-level target),
# then select/evaluate against held-out GT polygons. Predicted-class masks are diagnostic only.
UNET_PSEUDO_OUTPUT = OUTPUT_ROOT / f"unet_from_pseudo_{PIPELINE_PROFILE}"
RUN_TRAIN_UNET_FROM_PSEUDO = True
UNET_EPOCHS = 300
UNET_EARLY_STOP_PATIENCE = 25

UNET_PSEUDO_OUTPUT.mkdir(parents=True, exist_ok=True)
# Copy the OOM-killed run's training_log.csv into place BEFORE train_segmentation.py
# runs, so this run's epoch 76+ rows append to it instead of starting a fresh file --
# train_segmentation.py only writes a fresh header "if args.resume_from is None or
# not history_path.exists()" (see its main()), so the file must already exist here.
if RESUME_UNET_TRAINING_LOG_FROM:
    shutil.copy2(RESUME_UNET_TRAINING_LOG_FROM, UNET_PSEUDO_OUTPUT / "training_log.csv")
    print(f"Copied prior training_log.csv from {RESUME_UNET_TRAINING_LOG_FROM} into {UNET_PSEUDO_OUTPUT}")

unet_pseudo_cmd = [
    sys.executable, "train_segmentation.py", "--pipeline-profile", PIPELINE_PROFILE, "--data-root", str(BTXRD_ROOT),
    "--split-manifest", str(SPLIT_MANIFEST),
    "--train-split", "train", "--val-split", "val", "--image-size", str(IMAGE_SIZE),
    "--num-workers", str(NUM_WORKERS), "--epochs", str(UNET_EPOCHS),
    "--pos-weight-mode", "auto-clamped",
    "--early-stop-patience", str(UNET_EARLY_STOP_PATIENCE),
    "--train-pred-mask-root", str(TRAIN_GROUND_TRUTH_OUTPUT / "masks"),
    # Validation uses held-out polygon GT for checkpoint selection; pseudo masks are train-only.
    "--output-dir", str(UNET_PSEUDO_OUTPUT),
    "--multi-gpu",
]
# Resume from a prior (e.g. OOM-killed) run instead of restarting at epoch 1 --
# see Cell 1's RESUME_UNET_FROM. train_segmentation.py's --resume-from restores
# model/optimizer state and continues epoch numbering/training_log.csv.
if RESUME_UNET_FROM:
    unet_pseudo_cmd += ["--resume-from", str(RESUME_UNET_FROM)]
print(' '.join(unet_pseudo_cmd))
if RUN_TRAIN_UNET_FROM_PSEUDO:
    run_streaming(unet_pseudo_cmd)
else:
    print("RUN_TRAIN_UNET_FROM_PSEUDO=False; command prepared only.")

# Evaluate the trained U-Net against real GT polygons (not the pseudo-masks it trained on) --
# val gives development performance; test is the final held-out report after freezing.
UNET_EVAL_CSV = EVAL_OUTPUT / "unet_from_pseudo.csv"
UNET_EVAL_JSON = EVAL_OUTPUT / "unet_from_pseudo_summary.json"
unet_eval_cmd = [
    sys.executable, "evaluate_unet.py", "--data-root", str(BTXRD_ROOT),
    "--split-manifest", str(SPLIT_MANIFEST),
    "--split", FINAL_EVAL_SPLIT, "--checkpoint", str(UNET_PSEUDO_OUTPUT / "best_unet.pt"),
    "--image-size", str(IMAGE_SIZE), "--output-csv", str(UNET_EVAL_CSV), "--output-json", str(UNET_EVAL_JSON),
]
print(' '.join(unet_eval_cmd))
if RUN_TRAIN_UNET_FROM_PSEUDO:
    run_streaming(unet_eval_cmd)
    if UNET_EVAL_JSON.exists():
        display(pd.Series(json.loads(UNET_EVAL_JSON.read_text())).to_frame('value'))
else:
    print("Evaluation command prepared:", unet_eval_cmd)

# Optional controlled pos_weight ablation: canonical auto-clamped is above;
# these otherwise-identical runs compare raw ratio and fixed weight=10.
if RUN_POS_WEIGHT_ABLATION:
    for weight_label, weight_args in (
        ("raw", ["--pos-weight-mode", "auto-raw"]),
        ("fixed10", ["--pos-weight-mode", "manual", "--pos-weight-value", "10"]),
    ):
        ablation_output = OUTPUT_ROOT / f"unet_pos_weight_{weight_label}"
        command = [
            sys.executable, "train_segmentation.py", "--pipeline-profile", PIPELINE_PROFILE, "--data-root", str(BTXRD_ROOT),
            "--split-manifest", str(SPLIT_MANIFEST),
            "--train-split", "train", "--val-split", "val",
            "--image-size", str(IMAGE_SIZE), "--num-workers", str(NUM_WORKERS),
            "--epochs", str(UNET_EPOCHS),
            "--early-stop-patience", str(UNET_EARLY_STOP_PATIENCE),
            "--train-pred-mask-root", str(TRAIN_GROUND_TRUTH_OUTPUT / "masks"),
            "--output-dir", str(ablation_output), "--multi-gpu", *weight_args,
        ]
        run_streaming(command)
        run_streaming([
            sys.executable, "evaluate_unet.py", "--data-root", str(BTXRD_ROOT),
            "--split-manifest", str(SPLIT_MANIFEST), "--split", "val",
            "--checkpoint", str(ablation_output / "best_unet.pt"),
            "--image-size", str(IMAGE_SIZE),
            "--output-csv", str(EVAL_OUTPUT / f"unet_pos_weight_{weight_label}_val.csv"),
            "--output-json", str(EVAL_OUTPUT / f"unet_pos_weight_{weight_label}_val.json"),
        ])


## 10. Fully-supervised oracle baseline (Image -> GT -> U-Net)

This trains U-Net directly on ground-truth polygons -- no CAM, SAM, or pseudo-mask involved. It answers a
different question than the WSSS pipeline: what Dice/IoU is achievable at this image resolution/model/training
budget when full pixel-level supervision is available. It is **not** the WSSS result and must never be
reported as one; it exists purely as an upper-bound reference point for the comparison table in the next cell.


In [ ]:
# Cell 21 - fully-supervised oracle: train U-Net directly on GT polygons, then evaluate.
SUPERVISED_ORACLE_OUTPUT = OUTPUT_ROOT / "supervised_unet_oracle"
SUPERVISED_UNET_EPOCHS = 300
SUPERVISED_UNET_EARLY_STOP_PATIENCE = 25

SUPERVISED_ORACLE_OUTPUT.mkdir(parents=True, exist_ok=True)
# Same continuous-log copy as Cell 9b's U-Net (see its comment for why this
# must happen before train_segmentation.py runs).
if RESUME_SUPERVISED_ORACLE_TRAINING_LOG_FROM:
    shutil.copy2(RESUME_SUPERVISED_ORACLE_TRAINING_LOG_FROM, SUPERVISED_ORACLE_OUTPUT / "training_log.csv")
    print(f"Copied prior training_log.csv from {RESUME_SUPERVISED_ORACLE_TRAINING_LOG_FROM} into {SUPERVISED_ORACLE_OUTPUT}")

supervised_cmd = [
    sys.executable, "train_segmentation.py", "--pipeline-profile", PIPELINE_PROFILE, "--data-root", str(BTXRD_ROOT),
    "--split-manifest", str(SPLIT_MANIFEST),
    "--train-split", "train", "--val-split", "val", "--image-size", str(IMAGE_SIZE),
    "--num-workers", str(NUM_WORKERS), "--epochs", str(SUPERVISED_UNET_EPOCHS),
    "--pos-weight-mode", "auto-clamped",
    "--early-stop-patience", str(SUPERVISED_UNET_EARLY_STOP_PATIENCE),
    "--output-dir", str(SUPERVISED_ORACLE_OUTPUT),
    "--multi-gpu",
]
# Resume from a prior (e.g. OOM-killed) run instead of restarting at epoch 1 --
# see Cell 1's RESUME_SUPERVISED_ORACLE_FROM.
if RESUME_SUPERVISED_ORACLE_FROM:
    supervised_cmd += ["--resume-from", str(RESUME_SUPERVISED_ORACLE_FROM)]
print(' '.join(supervised_cmd))
if RUN_SUPERVISED_ORACLE_BASELINE:
    run_streaming(supervised_cmd)
else:
    print("RUN_SUPERVISED_ORACLE_BASELINE=False; no polygon labels enter WSSS generation.")

# Evaluate on the same split as WSSS. val is development-only; test is held out.
SUPERVISED_EVAL_CSV = EVAL_OUTPUT / "unet_supervised_oracle.csv"
SUPERVISED_EVAL_JSON = EVAL_OUTPUT / "unet_supervised_oracle_summary.json"
supervised_eval_cmd = [
    sys.executable, "evaluate_unet.py", "--data-root", str(BTXRD_ROOT),
    "--split-manifest", str(SPLIT_MANIFEST),
    "--split", FINAL_EVAL_SPLIT, "--checkpoint", str(SUPERVISED_ORACLE_OUTPUT / "best_unet.pt"),
    "--image-size", str(IMAGE_SIZE), "--output-csv", str(SUPERVISED_EVAL_CSV),
    "--output-json", str(SUPERVISED_EVAL_JSON),
]
print(' '.join(supervised_eval_cmd))
if RUN_SUPERVISED_ORACLE_BASELINE:
    run_streaming(supervised_eval_cmd)
    if SUPERVISED_EVAL_JSON.exists():
        display(pd.Series(json.loads(SUPERVISED_EVAL_JSON.read_text())).to_frame('value'))
else:
    print("Evaluation command prepared:", supervised_eval_cmd)


## 10b. WSSS pipeline vs. fully-supervised oracle -- side-by-side comparison

Both U-Nets were trained at the same image size and evaluated against the same split (`val` for development or held-out `test` for the final report)
(section 9b's `evaluate_unet.py` call and this section's, respectively) -- so `mean_tumor_dice`/`mean_tumor_iou`
below are directly comparable. The gap between the two rows is the observed performance gap between the complete WSSS pipeline and the fully supervised baseline
(WSSS) instead of full pixel-level polygon annotations (fully-supervised) for the exact same U-Net.


In [ ]:
# Cell 21a - freeze and verify before any locked-test access
TEST_PREDICTED_OUTPUT = OUTPUT_ROOT / f"pseudo_predicted_test_{PIPELINE_PROFILE}"
TEST_GROUND_TRUTH_OUTPUT = OUTPUT_ROOT / f"pseudo_ground_truth_test_{PIPELINE_PROFILE}"
TEST_PREDICTED_EVAL = EVAL_OUTPUT / "predicted_test.csv"
TEST_GROUND_TRUTH_EVAL = EVAL_OUTPUT / "ground_truth_test.csv"

def run_locked_test_protocol(output_dir, target_class, eval_csv):
    command = [
        sys.executable, "generate_pseudo_masks.py",
        "--pipeline-profile", PIPELINE_PROFILE, "--data-root", str(BTXRD_ROOT),
        "--split-manifest", str(SPLIT_MANIFEST), "--split", "test",
        "--frozen-config", str(FINAL_FROZEN_CONFIG),
        "--classifier-checkpoint", str(CLASSIFIER_CHECKPOINT),
        "--sam-checkpoint", str(SAM_CHECKPOINT), "--process-all",
        "--save-visuals-limit", "0", "--output-dir", str(output_dir),
    ]
    if target_class == "ground_truth":
        command += ["--cam-target-class", "ground_truth"]
    run_streaming(command)
    run_streaming([
        sys.executable, "evaluate_pseudo_masks.py",
        "--data-root", str(BTXRD_ROOT), "--split-manifest", str(SPLIT_MANIFEST),
        "--split", "test", "--frozen-config", str(FINAL_FROZEN_CONFIG),
        "--image-size", str(IMAGE_SIZE), "--pred-mask-root", str(output_dir / "masks"),
        "--output-csv", str(eval_csv),
    ])

if RUN_TEST_REPORT:
    # This block is the first and only place the notebook may access split=test.
    FINAL_FROZEN_CONFIG = OUTPUT_ROOT / "final_frozen_config.json"
    run_streaming([
        sys.executable, "tools/freeze_pipeline_config.py", "--profile", PIPELINE_PROFILE,
        "--split-manifest", str(SPLIT_MANIFEST),
        "--classifier-checkpoint", str(CLASSIFIER_CHECKPOINT),
        "--classifier-budget-audit", str(CLASSIFIER_BUDGET_AUDIT),
        "--sam-checkpoint", str(SAM_CHECKPOINT),
        "--unet-checkpoint", str(UNET_PSEUDO_OUTPUT / "best_unet.pt"),
        "--supervised-unet-checkpoint", str(SUPERVISED_ORACLE_OUTPUT / "best_unet.pt"),
        "--status", "final", "--output", str(FINAL_FROZEN_CONFIG),
    ])
    run_streaming([
        sys.executable, "tools/freeze_pipeline_config.py",
        "--output", str(FINAL_FROZEN_CONFIG), "--verify",
    ])

    run_locked_test_protocol(TEST_PREDICTED_OUTPUT, "predicted", TEST_PREDICTED_EVAL)
    run_locked_test_protocol(TEST_GROUND_TRUTH_OUTPUT, "ground_truth", TEST_GROUND_TRUTH_EVAL)
    for label, checkpoint in (
        ("wsss", UNET_PSEUDO_OUTPUT / "best_unet.pt"),
        ("supervised_oracle", SUPERVISED_ORACLE_OUTPUT / "best_unet.pt"),
    ):
        run_streaming([
            sys.executable, "evaluate_unet.py", "--data-root", str(BTXRD_ROOT),
            "--split-manifest", str(SPLIT_MANIFEST), "--split", "test",
            "--frozen-config", str(FINAL_FROZEN_CONFIG), "--checkpoint", str(checkpoint),
            "--image-size", str(IMAGE_SIZE),
            "--output-csv", str(EVAL_OUTPUT / f"{label}_test.csv"),
            "--output-json", str(EVAL_OUTPUT / f"{label}_test.json"),
        ])
else:
    print("RUN_TEST_REPORT=False; test split remains inaccessible.")


In [ ]:
# Cell 21b - build the final WSSS-vs-fully-supervised comparison table.
def load_unet_summary(path, label):
    if not path.exists():
        return {"pipeline": label, "status": "not run"}
    summary = json.loads(path.read_text())
    summary["pipeline"] = label
    summary["status"] = "complete"
    return summary

comparison_paths = (
    (EVAL_OUTPUT / "wsss_test.json", EVAL_OUTPUT / "supervised_oracle_test.json")
    if RUN_TEST_REPORT
    else (UNET_EVAL_JSON, SUPERVISED_EVAL_JSON)
)
comparison_rows = [
    load_unet_summary(comparison_paths[0], f"WSSS ({PIPELINE_PROFILE})"),
    load_unet_summary(comparison_paths[1], "Fully-supervised oracle (GT-trained)"),
]
comparison_df = pd.DataFrame(comparison_rows).set_index("pipeline")
display(comparison_df)

if "mean_tumor_dice" in comparison_df.columns and comparison_df["mean_tumor_dice"].notna().all():
    wsss_dice = comparison_df.loc[f"WSSS ({PIPELINE_PROFILE})", "mean_tumor_dice"]
    oracle_dice = comparison_df.loc["Fully-supervised oracle (GT-trained)", "mean_tumor_dice"]
    gap = oracle_dice - wsss_dice
    print(f"WSSS mean_tumor_dice={wsss_dice:.4f} | fully-supervised mean_tumor_dice={oracle_dice:.4f} "
          f"| gap={gap:.4f} ({gap / oracle_dice * 100:.1f}% relative to the oracle)")
else:
    print("Run both section 9b (WSSS U-Net) and section 10 (supervised oracle) cells above first.")


In [ ]:

# Cell 22 - reproducibility manifest and artifact completeness audit
required_artifacts = [
    CLASSIFIER_CHECKPOINT,
    CLASSIFIER_EVAL_OUTPUT / "summary.json",
    PREDICTED_OUTPUT / "pseudo_mask_manifest.csv",
    GROUND_TRUTH_OUTPUT / "pseudo_mask_manifest.csv",
    TRAIN_GROUND_TRUTH_OUTPUT / "pseudo_mask_manifest.csv",
    UNET_PSEUDO_OUTPUT / "best_unet.pt",
    UNET_EVAL_JSON,
]
if RUN_SUPERVISED_ORACLE_BASELINE:
    required_artifacts += [SUPERVISED_ORACLE_OUTPUT / "best_unet.pt", SUPERVISED_EVAL_JSON]
missing_artifacts = [str(path) for path in required_artifacts if not path.is_file()]
if missing_artifacts:
    raise FileNotFoundError(f"Notebook did not produce all required artifacts: {missing_artifacts}")

manifest = {
    "git_branch": GIT_ACTUAL_BRANCH,
    "git_commit": GIT_ACTUAL_COMMIT,
    "git_dirty": bool(GIT_DIRTY_LINES),
    "split_manifest": str(SPLIT_MANIFEST),
    "split_manifest_sha256": hashlib.sha256(SPLIT_MANIFEST.read_bytes()).hexdigest(),
    "classifier_checkpoint_sha256": hashlib.sha256(CLASSIFIER_CHECKPOINT.read_bytes()).hexdigest(),
    "sam_checkpoint_sha256": hashlib.sha256(SAM_CHECKPOINT.read_bytes()).hexdigest(),
    "python_version": sys.version,
    "torch_version": torch.__version__,
    "cuda_version": torch.version.cuda,
    "partition_unit": "heuristic case group; patient/lesion independence is not proven",
    "classifier_profile": PIPELINE_PROFILE,
    "canonical_wsss_train_target": "known image-level label",
    "predicted_class_protocol": "separate end-to-end diagnostic",
    "final_inference": "U-Net only",
    "development_evaluation_split": FINAL_EVAL_SPLIT,
    "locked_test_report_generated": RUN_TEST_REPORT,
    "frozen_config": str(FINAL_FROZEN_CONFIG) if RUN_TEST_REPORT else None,
    "test_tuning": False,
    "required_artifacts": [str(path) for path in required_artifacts],
}
(OUTPUT_ROOT / "notebook_artifacts.json").write_text(json.dumps(manifest, indent=2), encoding="utf-8")
display(pd.Series(manifest, name="value").to_frame())
print("Run All complete:", OUTPUT_ROOT)


## Interpretation checklist

- Report `predicted` as end-to-end inference and `ground_truth` only as localization/oracle diagnostics.
- Use conditional tumor Dice, end-to-end tumor Dice, normal empty-mask specificity, skipped count, and prompt-quality
  decomposition together; do not call macro-F1 a CAM metric.
- A low `foreground_recall`/high `support_loss_dice` indicates CAM or morphology failure. A good support but low
  `oracle_best_single_dice` indicates prompt/SAM candidate failure. A good oracle candidate but high
  `selection_loss_dice` indicates selection failure. A large `postprocess_delta_dice` indicates morphology failure.
- Polygon files are read only by evaluation/diagnostic cells and by the explicitly disabled supervised oracle baseline;
  no polygon-derived value is passed into CAM, prompts, candidate ranking, or WSSS post-processing.
- Keep the validation set for debugging and reserve the test split for one final locked report.
